# 41. Generating a CP toy with `generate_cp_toy`

**Objectives:**
- Build a B+/B- model pair that shares one `CPRealImag` coefficient.
- Generate a joint CP toy with `generate_cp_toy`, writing both charges to one ROOT file.
- Check the generated B+/B- event counts reflect the asymmetry implied by `dx`/`dy`.

Run cells top to bottom in a fresh kernel. Masses are in GeV, invariants in GeV², daughter
indices start at zero.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

import jax.numpy as jnp
import numpy as np
import uproot

from dalitzplotfitter import (
    CPRealImag, DecayChannel, DecayModel, NonResonant, Parameter, RealImag, Resonance,
    generate_cp_toy,
)

## 1. Build the two charge amplitudes from shared parameters

`CPRealImag` uses `c_q = (x + q*dx) + i*(y + q*dy)` with `q=+1` for B+ and `q=-1` for B-.
The nonzero deltas below are illustrative toy inputs, not experimental measurements. `x`/`y`/
`dx`/`dy` are built as floatable `Parameter.coefficient(...)` objects (not bare floats), as
required to later pass them to `FitSession.fit(start_values=...)`.

In [2]:
cp = CPRealImag(*[
    Parameter.coefficient(f"NR.{name}", value, owner="NR", bounds=bounds, step=0.02)
    for name, value, bounds in [
        ("x", 0.8, (-2, 2)), ("y", 0.3, (-2, 2)),
        ("dx", 0.35, (-1, 1)), ("dy", -0.25, (-1, 1)),
    ]
])

def make_model(parent, daughters, charge):
    return DecayModel(
        DecayChannel(parent, daughters),
        [Resonance("Kstar", (0, 2), RealImag(1, 0), mass=0.8958, width=0.0474, spin=1),
         NonResonant(cp.for_charge(charge), name="NR")],
        normalization_method="square-dalitz", normalization_pair=(0, 2),
        normalization_resolution=100,
    )

plus_model = make_model("B+", ("K+", "pi+", "pi-"), +1)
minus_model = make_model("B-", ("K-", "pi-", "pi+"), -1)
truth = {p.name: p.value for p in plus_model.parameters}

## 2. Generate the joint CP toy

`generate_cp_toy`'s docstring notes that the signal (and any background) charge split uses a
deterministic *accepted-integral* convention: `n_plus ~ Binomial(size, I_plus / (I_plus + I_minus))`
with `I_q` the accepted signal integral for charge `q`, so the expected split follows the ratio
of the two coherent integrals, not a fixed 50/50 draw. `output_root=` optionally saves both
charges into one ROOT file/tree, distinguished by a signed `charge` branch, as documented in
`docs/toy_generation.md`.

In [3]:
root_path = "tutorial_41_cp_toy.root"
plus_data, minus_data = generate_cp_toy(
    plus_model, minus_model, 6000, parameters=truth, seed=41,
    inverse_resolution=384, include_momenta=False,
    output_root=root_path,
)
print("Generated B+ events:", plus_data.size)
print("Generated B- events:", minus_data.size)
assert plus_data.size + minus_data.size == 6000

Generated B+ events: 3727
Generated B- events: 2273


## 3. Check the charge asymmetry against the coherent integrals

Compute `I_plus`/`I_minus` directly from `intensity` on each model's own normalization sample
(the same accepted-integral quantity the generator uses internally) and compare the predicted
split to the observed event counts.

In [4]:
def accepted_integral(model):
    sample = model.normalization_sample
    return float(jnp.mean(sample.weights * model.intensity(sample.as_dict(), truth)))

i_plus = accepted_integral(plus_model)
i_minus = accepted_integral(minus_model)
expected_plus_fraction = i_plus / (i_plus + i_minus)
observed_plus_fraction = plus_data.size / (plus_data.size + minus_data.size)
print(f"I_plus={i_plus:.4f}  I_minus={i_minus:.4f}")
print(f"Expected B+ fraction: {expected_plus_fraction:.3f}")
print(f"Observed B+ fraction: {observed_plus_fraction:.3f}")

# Binomial standard error at this sample size, as a sanity tolerance for the check below.
n_total = plus_data.size + minus_data.size
binomial_se = float(np.sqrt(expected_plus_fraction * (1 - expected_plus_fraction) / n_total))
assert abs(observed_plus_fraction - expected_plus_fraction) < 5 * binomial_se, (
    "observed charge split deviates from the accepted-integral expectation by more than 5 sigma"
)
print(f"Binomial std error: {binomial_se:.4f} -- observed split matches the integral asymmetry.")

I_plus=2.3250  I_minus=1.5050
Expected B+ fraction: 0.607
Observed B+ fraction: 0.621
Binomial std error: 0.0063 -- observed split matches the integral asymmetry.


## 4. Confirm the ROOT file round-trips the charge split

In [5]:
with uproot.open(root_path) as f:
    tree = f["DecayTree"]
    charge = tree["charge"].array(library="np")
print("Events with charge > 0 (B+):", int(np.sum(charge > 0)))
print("Events with charge < 0 (B-):", int(np.sum(charge < 0)))
assert int(np.sum(charge > 0)) == plus_data.size
assert int(np.sum(charge < 0)) == minus_data.size

Events with charge > 0 (B+): 3727
Events with charge < 0 (B-): 2273


## Summary and exercises

1. `generate_cp_toy` splits events between charges using the accepted-integral ratio
   `I_plus / (I_plus + I_minus)`, not the raw `dx`/`dy` values directly -- the two are related
   through the coherent amplitude and the normalization convention, not a simple formula.
2. Setting `dx = dy = 0` makes the two models identical up to charge-conjugate kinematics and
   the expected split collapses to 50/50.
3. `output_root=` writes both charges to one file/tree with a signed `charge` branch
   (`docs/toy_generation.md`); omit it to keep everything in memory only.
4. Try larger `|dx|`/`|dy|` and re-run to see a more pronounced charge asymmetry.

Reference: [CP conventions](../../docs/cp_coefficients.md), [toy generation](../../docs/toy_generation.md).

Return to [the course guide](TUTORIALS.md).